In [9]:
import time

import numpy as np

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader

# Evaluate Model

In [10]:
def evaluate_model(model, test_loader, device, criterion, test_dataset_size):
    since = time.time()
    model.eval()
    running_loss = 0.0
    running_corrects = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            running_corrects += (labels == predicted).sum().item()

    epoch_loss = running_loss / test_dataset_size
    epoch_accuracy = 100 * running_corrects / test_dataset_size

    print(f"Eval Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%")
    print(f'Got {running_corrects} out of {test_dataset_size} images correctly')
    
    time_elapsed = time.time() - since
    print(f'Evaluation complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    return epoch_loss, epoch_accuracy

# Practice Test Dataset from Kaggle

In [11]:
# Dataset from https://www.kaggle.com/datasets/shifatearman/bananalsd?resource=download

# Get dataset here https://drive.google.com/drive/folders/1w6j_dpi9ufCIHuqg5Ka_eCrRLIyG-KTc?usp=sharing

# Original + Augmented
test_dataset_path = './prac_test/'

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_dataset = torchvision.datasets.ImageFolder(
    root=test_dataset_path, 
    transform=test_transform
)

test_loader = torch.utils.data.DataLoader(  
    dataset=test_dataset, 
    batch_size=32, 
    shuffle=True
)

# Define the model architecture
resnet152_model = models.resnet152(weights='DEFAULT')
num_features = resnet152_model.fc.in_features
num_classes = 4
resnet152_model.fc = nn.Linear(num_features, num_classes)

# Load the trained weights
resnet152_trained_model = "CNN_transfer_learning/models/resnet152_1.pth"
state_dict = torch.load(resnet152_trained_model) 
resnet152_model.load_state_dict(state_dict)

# Evaluate model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")
test_dataset_size = len(test_dataset)
loss_function = nn.CrossEntropyLoss()
loss, accuracy = evaluate_model(resnet152_model, test_loader, device, loss_function, test_dataset_size)

Using cpu device
Eval Loss: 0.1557, Accuracy: 97.83%
Got 2482 out of 2537 images correctly
Evaluation complete in 4m 57s
